DAY 3

In [ ]:
%pip install -q langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="openai/gpt-oss-120b", temperature=0.3, api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! 😊 Yes, we offer eggless versions of all our cakes—plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!
Your name is Ravi.


In [17]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! 😊 Yes, we offer eggless versions of all our cakes—plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi.')]

In [19]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")



In [21]:
run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500. Let me know if you’d like to add anything el
FAIL | Do you make eggless cakes? → Yes! All our cakes – plum, chocolate and cupcakes – are available in e
FAIL | Can I get a cake today evening? → I’ll check with the owner and get back to you shortly!
PASS | Do you deliver to Kakinada? → I'll check with the owner and let you know!
PASS | What is the price of 6 cupcakes? → Sure! Six cupcakes would cost ₹480. Let me know if you’d like to add a

3/5 passed (60%)


In [22]:
from langchain.agents import create_agent

agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result_order = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes! All our cakes – plum, chocolate and cupcakes – are available in egg‑less versions. Let us know which one you’d like!
